# 1차 기초모델

In [5]:
%pip install pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install jinja2

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 1. 데이터 불러오기

본 데이터는 개발용 DB에서 2026년 9월 15일 추출한 것으로, 운영 환경의 전체 상품과는 차이가 있을 수 있다.

In [7]:
import pandas as pd

column_names = ['id', 'supplier_id', 'name', 'part_type', 'category', 'split_group']
df = pd.read_csv('./data/supplier_part_raw.csv', header=None, names=column_names, keep_default_na=False)

print("전체:", df.shape)
print(df["split_group"].value_counts())
print("카테고리 수:", df[df["split_group"] == "train"]["category"].nunique())
print("공급사 수:", df["supplier_id"].nunique())

df

전체: (21631, 6)
split_group
train        19955
unlabeled     1097
etc            502
old             77
Name: count, dtype: int64
카테고리 수: 213
공급사 수: 109


,id,supplier_id,name,part_type,category,split_group
0,7132,130,IPBS-E120 대체,DOMESTIC_VEHICLE,브레이크패드센서,train
1,6923,37,브레이크 디스크 앞 (1개),DOMESTIC_VEHICLE,디스크로터,train
2,4772,139,분류코드 BB 대체,DOMESTIC_VEHICLE,점화플러그,train
3,6932,67,브레이크 디스크 앞,DOMESTIC_VEHICLE,디스크로터,train
4,4774,19,스파크 플러그 X,DOMESTIC_VEHICLE,점화플러그,train
...,...,...,...,...,...,...
21626,109334,33,텐셔너베어링 DJ 34302-5,IMPORTED_VEHICLE,텐셔너,train
21627,109335,33,텐셔너베어링 DJ 34302-4,IMPORTED_VEHICLE,텐셔너,train
21628,109336,33,텐셔너베어링 DJ 34302-3,IMPORTED_VEHICLE,텐셔너,train
21629,109337,33,텐셔너베어링 DJ 34302-2,IMPORTED_VEHICLE,텐셔너,train


## 2. 데이터 정제
학습에 쓸 수 없거나 판단에 방해가 되는 요소를 제거한다.
1. 특수기호 제거
2. 메모성 상품명 제거

### 2-1. 특수기호 조사
한글·영문·숫자·기본 구두점 외의 문자를 전수 조사하여 사용 실태를 확인한다.

In [8]:
train = df[df["split_group"] == "train"]
train["name"].sample(30, random_state=1).tolist()

['앞 허브베어링',
 '로워암',
 'BLADE',
 '라이닝 베스타,프레지오,봉고3후',
 '◆◆점화 코일 ZS-K 4X1 (IC049)',
 '5PK1240 벨트',
 '토스카 L6 2.5 SET 6PK1835',
 '앞패드 스팅어 하드론Z(18인치)',
 '브레이크마스터실린더 CBS MT',
 '부동통 캡',
 'IPB-E159R 대체',
 '앞로워암 좌',
 '에어필터 S80 2.5 06~',
 '씰리콘 건(국산) 원통형',
 '스파크 플러그 X',
 '4PK829 벨트',
 '폭스바겐 앞패드(센서포함)',
 'ELRING 오일 팬 가스켓',
 '에어필터 2071.6HDI 06~',
 'IPB-E038R 대체',
 '300 2.7~5.7 08~10 (160A)',
 '트레일러-SW335 엑슬 후드럼 (진도 H10)',
 'ELRING 가스켓 흡기 다기관',
 '앞패드 그랜드 스타렉스 16년형',
 '휠스피드센서',
 'Pressure converter',
 'COOLANT HOSE',
 'IPCA-E259C 대체',
 'A32.8 벨트',
 'HELLA 오일 압력 스위치']

#### 특수기호 조사 및 예시 출력

In [9]:
import re
from collections import Counter

special_char = Counter()
for name in df["name"]:
    for ch in name:
        if not re.match(r"[가-힣a-zA-Z0-9\s\.\,\-\/\(\)\[\]~%&+:;'\"]", ch):
            special_char[ch] += 1

for ch, cnt in special_char.items():
    examples = df[df["name"].str.contains(ch, regex=False)]["name"].head(3).tolist()
    print(f"{repr(ch)}  ({cnt}건)")
    for e in examples:
        print("   ", e)
    print()

special_list = list(special_char.keys())
print(special_list)

'▲'  (370건)
    ▲▲PT Cruiser2.0 00~10(120A)
    ▲▲A6 2.4 98~05 (120A)
    ▲▲Cooper 04~ (100A)

'{'  (335건)
    {B} TENSIONER
    {A} TENSIONER
    {Z} THERMOSTAT ASSY

'}'  (336건)
    {B} TENSIONER
    {A} TENSIONER
    {Z} THERMOSTAT ASSY

'ㅗ'  (2건)
    NF 시가코일 짹(ㅗ)
    점화코일 뉴에쿠스/오피러스(ㅗ)

'☆'  (3건)
    300C 2.7~6.1 04~12 (160A)☆
    VOYAGER 2.5 CRD 00~08 (160A)☆
    300C 3.6 GAS 11~13 (180A)☆

'㉭'  (458건)
    ㉭포르쉐 앞패드
    ㉭크라이슬러 앞패드
    ㉭패드센서 1635401717

'Ⅳ'  (2건)
    Aero twn(530/475) : Golf Ⅳ (v
    Aero twn(530/475) : Golf Ⅳ (v

'★'  (29건)
    와이퍼 블레이드 (600/475)★
    와이퍼 블레이드 (650/475)★
    와이퍼 블레이드 (600/550) ★

'_'  (8건)
    앞패드 벤츠 C200_TEST
    에어필터_TEST
    에어필터_테스트

'大'  (13건)
    휴즈 大 10A
    휴즈 大 5A
    아반테,베르나 1.5 DO 텐션(大)

'Ⅱ'  (24건)
    앞패드크레도스Ⅱ,카렌스 하드론
    앞패드 크레도스Ⅱ,카렌스 단종
    뒤패드 소나타Ⅱ,마르샤(단종)

'◆'  (316건)
    ◆◆오일 필터 A6(4A) 2.0 01~05
    ◆◆+34 WR8LC X
    ◆◆스파크 플러그 SMART

'小'  (14건)
    세피아,스펙트라 DO 텐션(小)
    아반테 텐션(小)
    포터,스타렉스,테라칸2.5 텐션小

'無'  (2건)
    EF 파워텐션(볼트無)영업용

#### 중괄호 표기 검증

In [10]:
# 중괄호 표기가 부품 종류와 무관한지 확인
brace = df[df["name"].str.contains("{", regex=False)]
brace_letter = brace["name"].str.extract(r"\{(.)\}")[0]
brace_name = brace["name"].str.replace(r"\{.\}\s*", "", regex=True)

check = pd.DataFrame({"letter": brace_letter, "name": brace_name,
                      "category": brace["category"]})

check.groupby("name").agg(
    문자=("letter", lambda x: sorted(set(x))),
    카테고리=("category", lambda x: sorted(set(x))),
    건수=("letter", "size")
).sort_values("건수", ascending=False).head(10)

,문자,카테고리,건수
name,,,
GASKET,"[A, B, M, V, X]","[EGR밸브, 라디에이터호스, 미션오일팬, 배기매니폴드, 써모스탯, 엔진오일쿨러, ...",33
HOSE,"[A, B, M]","[라디에이터호스, 배기파이프, 연료호스/파이프, 흡기매니폴드]",11
THERMOSTAT,"[A, B, M, X]",[써모스탯],8
SEAL RING,"[A, M]","[라디에이터호스, 스로틀바디, 써모스탯, 엔진오일쿨러, 터보차저/인터쿨러, 흡기매니폴드]",7
SEAL,"[A, B]","[리데나, 벨브트로닉, 흡기매니폴드]",5
FUEL HOSE,[M],[연료호스/파이프],5
OIL FILTER,"[B, M]",[오일필터],5
TENSIONER,"[A, B, M]","[, 텐셔너]",4
CONNECTOR,"[B, M]","[라디에이터호스, 쇼크업소버]",4


#### 대체품 메모 형태 확인

In [11]:
# 대체품 메모 형태 확인 — 행 제거 여부 판단
arrow = df[df["name"].str.contains("=>", regex=False)]
print("건수:", len(arrow))
arrow[["name", "category"]]

건수: 5


,name,category
20626,"(R1049=>B)마이티1,2(고상)/카운티/파맥스2.5 드럼(전/후)",브레이크드럼
20629,(R1052=>B)카운티(신형) 드럼(전/후),브레이크드럼
20672,(R2025=>B)프론티어 1톤 후 드럼,브레이크드럼
20696,(R1053=>B)포타(96) 드럼(후),브레이크드럼
20751,(R1051=>B)현대3.5톤/파맥스3.5톤 드럼(전/후),브레이크드럼


-----------

### 2-2. 특수기호 처리 기준
전수 조사 결과 26종의 특수기호를 발견하였다.

**부품 카테고리 판단에 기여하는가**를 기준으로 검토한 결과, 특수기호 전체를 제거 대상으로 판단하였다.

| 유형 | 기호 | 건수 | 예시 |
|---|---|---|---|
| 강조·분류 표시 | `▲` `◆` `㉭` `★` `☆` | 1,176 | `▲▲PT Cruiser2.0 (120A)` |
| 공급 경로 구분 | `{` `}` | 671 | `{A} TENSIONER` |
| 차종 세대 표기 | `Ⅱ` `Ⅲ` `Ⅳ` `Ⅸ` | 31 | `크레도스Ⅱ`, `Golf Ⅳ` |
| 규격·구성 표기 | `大` `小` `無` `有` `°` `Ø` `*` | 45 | `휴즈 大 10A`, `볼트無` |
| 제조사 번호·메모 | `#` `=` `>` | 66 | `(R1049=>B)`, `#16` |
| 기타 | `_` `ㅗ` `凸` `` ` `` `é` | 14 | `C200_TEST` |

**판단 근거**

- 규격·구성 표기(`大`, `無` 등)는 같은 카테고리 내 세부 상품을 구분하는 정보로,
  카테고리 판단에는 기여하지 않아 함께 제거한다.
- `{A} THERMOSTAT`과 `{B} THERMOSTAT`이 동일 카테고리(써모스탯)로 지정된 것을 확인하여,
  중괄호 표기가 부품 종류와 무관함을 검증하였다.

**처리 방식**

기호를 제거한 뒤 **남는 문자가 의미를 갖는지**에 따라 처리 방식을 구분하였다.

| 방식 | 대상 | 처리 전 → 후 |
|---|---|---|
| 기호만 치환 | 대부분의 기호 | `▲▲PT Cruiser2.0` → `PT Cruiser2.0` |
| 덩어리째 제거 | `{알파벳}`, `(코드=>코드)` | `{A} TENSIONER` → `TENSIONER`<br>`(R1049=>B)마이티 드럼` → `마이티 드럼` |

`{A}`의 경우 기호만 제거하면 `A TENSIONER`처럼 무의미한 문자가 남으므로 내용을 포함해 제거한다.

`(R1049=>B)`는 대체품 안내 메모이나, 해당 5건 모두 메모 뒤에 정상적인 부품명이 이어지고
카테고리도 브레이크드럼으로 올바르게 지정되어 있었다. 따라서 행을 제거하지 않고
메모 부분만 제거한다.

In [12]:
# 1) 덩어리째 제거: {알파벳}, (코드=>코드)
df["name_clean"] = (df["name"]
                    .str.replace(r"\{[^}]*\}", " ", regex=True)
                    .str.replace(r"\([^)]*=>[^)]*\)", " ", regex=True))

# 2) 나머지 특수기호를 공백으로 치환하고 연속 공백 정리
pattern = "[" + "".join(re.escape(ch) for ch in special_list) + "]"
df["name_clean"] = (df["name_clean"]
                    .str.replace(pattern, " ", regex=True)
                    .str.replace(r"\s+", " ", regex=True)
                    .str.strip())

changed = df[df["name"] != df["name_clean"]]
print("변경된 행:", len(changed))

변경된 행: 1524


In [13]:
# 전체 정제 결과 확인
changed[["name", "name_clean"]].sample(15, random_state=1)

,name,name_clean
10025,㉭벤츠 뒤패드,벤츠 뒤패드
11984,{A} HOSE,HOSE
9901,{M} SPRING,SPRING
17074,{B} CREW CAP FOR EXPANSION,CREW CAP FOR EXPANSION
5064,㉭패드센서 34356870352,패드센서 34356870352
9264,{A} GASKET,GASKET
2505,\n항균필터 \nHUMMER H2 6.0 03~07,항균필터 HUMMER H2 6.0 03~07
6618,㉭에어필터 PT CRUISER 2.4 06~1,에어필터 PT CRUISER 2.4 06~1
760,▲▲C30 1.6/2.0D 06~ (150A),C30 1.6/2.0D 06~ (150A)
9825,㉭크라이슬러 뒤패드,크라이슬러 뒤패드


In [14]:
# 덩어리 제거 확인
df[df["name"].str.contains(r"\{", regex=True)][["name", "name_clean"]].head(10)

,name,name_clean
193,{B} TENSIONER,TENSIONER
195,{A} TENSIONER,TENSIONER
200,{Z} THERMOSTAT ASSY,THERMOSTAT ASSY
942,{B} OIL FILTER,OIL FILTER
1451,{B} COVER PLATE,COVER PLATE
1552,{M} T/M BOLT,T/M BOLT
1604,{M} BRAKE PAD,BRAKE PAD
1605,{M} BRAKE PADS,BRAKE PADS
1607,{M} TPMS SENSOR,TPMS SENSOR
1657,"{M} RING,GENERAL,M","RING,GENERAL,M"


In [15]:
df[df["name"].str.contains(r"\=>", regex=True)][["name", "name_clean"]].head(10)

,name,name_clean
20626,"(R1049=>B)마이티1,2(고상)/카운티/파맥스2.5 드럼(전/후)","마이티1,2(고상)/카운티/파맥스2.5 드럼(전/후)"
20629,(R1052=>B)카운티(신형) 드럼(전/후),카운티(신형) 드럼(전/후)
20672,(R2025=>B)프론티어 1톤 후 드럼,프론티어 1톤 후 드럼
20696,(R1053=>B)포타(96) 드럼(후),포타(96) 드럼(후)
20751,(R1051=>B)현대3.5톤/파맥스3.5톤 드럼(전/후),현대3.5톤/파맥스3.5톤 드럼(전/후)


### 2-3. 메모성 상품명 및 테스트 데이터 제거

상품명에 부품 정보가 아닌 내용이 입력된 경우를 확인한다.

In [16]:
NOTICE = re.compile(r"대체|단종|출고정지|출고중지|소진|분류코드|분류BB|TEST|테스트",
                    re.IGNORECASE)

train_all = df[df["split_group"] == "train"]
notice_rows = train_all[train_all["name_clean"].str.contains(NOTICE)]

print("제거 대상:", len(notice_rows))
display(notice_rows["name_clean"].sample(20, random_state=1).tolist())

print("\n[테스트 데이터]")
train_all[train_all["name_clean"].str.contains("테스트|TEST", case=False)][["name_clean", "category"]]

제거 대상: 874


['앞패드 카니발 04년형(단종)',
 'IPB-E003F 대체',
 'IPO-319 로 대체',
 'IPBS-E143 대체',
 'IPA-P337 대체',
 '동일품번 보쉬코리아로 대체 X',
 'IPBS-E037 대체',
 'IPA-P204 로 대체',
 '97123-257051AC 대체',
 '연료필터 무쏘,뉴코란도(단종)',
 'A6 3.0 TDI 03~ (180A)단종',
 'IPA-486 로 대체',
 '출고정지 형상 다름',
 'IPSP-A007 대체',
 '스파크 플러그(단종)',
 'IPA-P206 대체',
 '18812-0242236663 대체',
 'IPA-P400 로 대체',
 'IPCA-214C 또는 IPCA-E329C 대체',
 'IPCA-E119C 대체']


[테스트 데이터]


,name_clean,category
1156,테스트5,실린더블록
2902,앞패드 벤츠 C200 TEST,수입기타
2903,에어필터 TEST,수입기타
2904,부품테스트,수입기타
2935,부품테스트5,수입기타
2958,부품테스트4,수입기타
2959,에어필터 테스트,수입기타
2999,부품 test008,수입기타
3000,부품 test009,수입기타
20998,테스트3,실린더블록


**처리 기준**

상품명에 부품 정보가 아닌 내용이 입력된 경우 상품명만으로 부품 종류를 판별할 수 없으므로 학습과 평가에서 제외한다.

| 유형 | 예시 | 성격 |
|---|---|---|
| 대체품 안내 | `IPEO-761로 대체`, `분류코드 BB로 대체` | 상품명 자리의 재고 메모 |
| 재고·출고 상태 | `할인판매 재고소진후 단종`, `출고정지 형상 다름` | 상품명 자리의 상태 표시 |
| 테스트 데이터 | `테스트5`, `부품 test008`, `에어필터 테스트` | 개발 과정에서 생성된 데이터 |

**테스트 데이터 제외 근거**

본 데이터는 개발용 DB에서 추출한 것으로, 부품 등록·수정 기능을 테스트하는 과정에서
생성된 상품이 포함되어 있다. 해당 18건은 카테고리가 실린더블록, 밸브바디, 수입기타 등으로
임의 지정되어 있어 실제 거래 상품이 아님을 확인하였다.

**제거 결과**

| 구분 | 건수 |
|---|---|
| 정답 카테고리 있음 | 19,955 |
| 제거 (메모성 + 테스트) | 874 |
| **학습 대상** | **19,081** |

> 제거한 874건은 상품명에 부품 정보가 없어 어떤 모델로도 분류할 수 없다.
> 따라서 이후 보고하는 성능은 **분류 가능한 19,081건에 대한 성능**이며,
> 추출된 데이터셋 전체에 대한 성능이 아님에 유의한다.

In [17]:
train = df[df["split_group"] == "train"].copy()

is_notice = train["name_clean"].str.contains(NOTICE)
print("제거:", is_notice.sum())

train = train[~is_notice]
print("남은 건수:", len(train))

제거: 874
남은 건수: 19081


## 3. 중복 제거와 학습 대상 확정

동일한 상품명이 학습과 평가 양쪽에 포함되면 모델이 이미 본 답을 맞히는 셈이 되어 성능이 과대평가된다(데이터 누수).

상품명 기준으로 중복을 정리한 뒤, 학습에 사용할 카테고리 범위를 확정한다.

### 3-1. 중복 규모 확인
상품명 전체가 똑같은 행을 찾는다.


In [18]:
unique_name_count = train["name_clean"].nunique()
duplicate_rate = 1 - unique_name_count / len(train)

print("전체 건수:", len(train))
print("고유 상품명:", unique_name_count)
print(f"중복 비율: {duplicate_rate:.1%}")

train["name_clean"].value_counts().head(10)

전체 건수: 19081
고유 상품명: 11736
중복 비율: 38.5%


name_clean
브레이크 디스크 앞    244
호스            230
발전기           197
브레이크 디스크 뒤    190
에어필터          183
워터펌프          142
텐셔너           122
라디에이터         119
스타터           102
연료필터           91
Name: count, dtype: int64

### 3-2. 라벨 충돌 확인
동일한 상품명에 서로 다른 카테고리가 지정된 경우를 확인한다.

상품명이 너무 범용이라 상품명만으로 특정할 수 없는 경우가 있다.

In [19]:
conflict = train.groupby("name_clean")["category"].nunique()
conflict_names = conflict[conflict > 1].index

print("카테고리가 충돌하는 상품명:", len(conflict_names))

train[train["name_clean"].isin(conflict_names)] \
    .groupby("name_clean")["category"].unique().head(20)

카테고리가 충돌하는 상품명: 142


name_clean
ABS 센서                                                         [휠스피드센서, ABS센서]
ACTUATOR                                                     [배기파이프, 도어락액추에이터]
ADAPTER                                                        [써모스탯, 라디에이터호스]
AIR DUCT                                                        [언더커버, 흡기매니폴드]
AIR PIPE                                                     [흡기매니폴드, 라디에이터호스]
AUDI GASKET                                           [크랭크케이스, 라디에이터호스, 세퍼레이트]
AUDI HOSE                                                    [흡기매니폴드, 라디에이터호스]
BEARING                                                   [휠허브, 쇼크업소버, 크랭크샤프트]
BELLOW                                                       [쇼크업소버, 스티어링기어박스]
BOWDEN CABLE                                                          [도어, 보닛]
BRACKET                                      [브라켓, 밸브&스프링, 쇼크업소버, 선바이저, 배기파이프]
BRAKE SWITCH                                                  [내부스위치, 브레이크스위치]
BUSHING                                  

### 3-3. 중복 제거

상품명 하나당 한 행만 남긴다. 카테고리가 충돌하는 142개 상품명은 제외한다.

**제외 근거**

충돌 142건 중 **67건(47%)이 동점**이어서, 다수결을 적용해도 사실상 무작위 선택이 된다.
또한 `CLIP`(8개 카테고리), `BRACKET`(5개)처럼 여러 부품에 실제로 사용되는 범용 명칭이
대부분이라 특정 카테고리를 정답으로 지정할 근거가 없다.

2단계에서 재고 메모를 제거한 것과 같은 기준이다.
상품명만으로 사람도 판별할 수 없는 데이터는 학습과 평가 모두에서 제외한다.

**다수결 적용 시와 비교 (5회 반복, 로지스틱 회귀)**

| 처리 | 정확도 | Macro-F1 |
|---|---|---|
| 다수결 | 92.1% ± 0.4% | 0.859 |
| 제외 | 92.9% ± 0.2% | 0.872 |

제외 시 성능이 소폭 상승하고 편차가 절반으로 줄었다. 모델이 개선된 것이 아니라,
정답이 임의로 정해진 문제가 평가에서 빠졌기 때문이다.
데이터 손실은 142개(고유 상품명의 1.2%)에 그친다.

In [20]:
# 카테고리가 충돌하는 상품명은 제외
conflict = train.groupby("name_clean")["category"].nunique()
conflict_names = conflict[conflict > 1].index
print("카테고리가 충돌하는 상품명:", len(conflict_names), "(제외)")

kept = train[~train["name_clean"].isin(conflict_names)]

data_all = kept.groupby("name_clean").agg(
    category=("category", "first"),
    supplier_id=("supplier_id", lambda x: Counter(x).most_common(1)[0][0]),
).reset_index().rename(columns={"name_clean": "name"})

print("고유 상품명:", len(data_all))

카테고리가 충돌하는 상품명: 142 (제외)
고유 상품명: 11594


### 3-4. 학습 대상 카테고리 확정

상품명이 적은 카테고리는 학습과 평가 모두에 충분한 예시를 확보하기 어렵다.

상품명을 학습용 80%, 평가용 20%로 분할할 때 카테고리당 평가용 상품명이 최소 4개가 확보되도록 **20개**를 기준으로 설정한다.

> 기준값 20이 적절한지는 7-5에서 실험으로 확인한다.

In [21]:
KEEP_THRESHOLD = 20

counts = data_all["category"].value_counts()
keep_categories = counts[counts >= KEEP_THRESHOLD].index

data = data_all[data_all["category"].isin(keep_categories)].reset_index(drop=True)

print("전체 카테고리:", len(counts))
print("학습 대상 카테고리:", len(keep_categories))
print("학습 대상 상품명:", len(data))
print(f"커버리지: {len(data) / len(data_all):.1%}")

전체 카테고리: 208
학습 대상 카테고리: 74
학습 대상 상품명: 10826
커버리지: 93.4%


## 4. 학습-평가 데이터 분할

같은 데이터로 학습하고 평가하면 성능을 제대로 측정할 수 없으므로,
고유 상품명을 학습용 80%와 평가용 20%로 나눈다.

두 가지 방식으로 나누어 서로 다른 것을 확인한다.

| 방식 | 나누는 기준 | 확인하려는 것 |
|---|---|---|
| A. 무작위 분할 | 상품명을 섞어서 8:2 | 기본 성능 |
| B. 공급사 단위 분할 | 공급사를 통째로 8:2 | 처음 보는 공급사의 상품명에도 통하는가 |

### 4-1. 공급사 편중 확인

B 방식이 필요한 이유를 먼저 확인한다.

In [22]:
supplier_per_category = data.groupby("category")["supplier_id"].nunique()

single_supplier_categories = supplier_per_category[supplier_per_category == 1].index
two_supplier_categories = supplier_per_category[supplier_per_category == 2].index

print("단일 공급사 카테고리:", len(single_supplier_categories))
print(list(single_supplier_categories))
print("\n2개 공급사 카테고리:", len(two_supplier_categories))
print(list(two_supplier_categories))

단일 공급사 카테고리: 4
['레베어링', '릴리스실린더', '클러치마스터실린더', '휠실린더']

2개 공급사 카테고리: 9
['마스터실린더', '몰딩/트림', '범퍼', '브레이크드럼', '사이드미러', '언더커버', '점화케이블', '팽창벨브', '휠하우징커버']


In [23]:
single_supplier_categories = supplier_per_category[supplier_per_category == 1].index

for cat in single_supplier_categories:
    print(cat)
    print(data[data["category"] == cat]["name"].head(8).tolist())
    print()

레베어링
['클러치릴리즈베어링 DJ 3400 B-03', '클러치릴리즈베어링 DJ 3400 BS A-2', '클러치릴리즈베어링 DJ 34302-11', '클러치릴리즈베어링 DJ 34302-17', '클러치릴리즈베어링 DJ 34302-18', '클러치릴리즈베어링 DJ 34302-19', '클러치릴리즈베어링 DJ 34302-20', '클러치릴리즈베어링 DJ 34302-29']

릴리스실린더
['릴리스 실린더 쏘렌토', '릴리스실린더 EF,투싼,옵티마,XG', '릴리스실린더 E마이티 NARROW', '릴리스실린더 XG,EF,리갈,옵티마', '릴리스실린더 갤로퍼 인터쿨러', '릴리스실린더 그랜드스타렉스', '릴리스실린더 그레이스,갤로퍼', '릴리스실린더 뉴프라이드,HD']

클러치마스터실린더
['클러치마스터 포터97 (AU) T2', '클러치마스터/뉴라이노', '클러치마스터실린더 4.5톤/5톤', '클러치마스터실린더 E 마이티', '클러치마스터실린더 EF쏘나타', '클러치마스터실린더 EG5톤(구형)', '클러치마스터실린더 E에어로타운', '클러치마스터실린더 NF쏘나타']

휠실린더
['휠실린더 더넥스트 스파크', '휠실린더 올뉴마이티 전우 상/하', '휠실린더 올뉴마이티 전좌 상/하', '휠실린더 올뉴마이티 후우 상/하', '휠실린더 올뉴마이티 후좌 상/하', '휠실린더 전 우 뉴라이노', '휠실린더 전 우 마이티', '휠실린더 전 우 마이티2.5톤']



학습 대상 74개 카테고리 중 **4개는 단일 공급사**, **9개는 두 공급사**의 상품으로만 구성된다.

단일 공급사 카테고리의 상품명을 확인한 결과, 표기 형식이 일정하게 반복되고 있었다.

| 카테고리 | 상품명 형식 | 예시 |
|---|---|---|
| 레베어링 | `클러치릴리즈베어링 DJ` + 번호 | `클러치릴리즈베어링 DJ 34302-11` |
| 릴리스실린더 | `릴리스실린더` + 차종 | `릴리스실린더 그랜드스타렉스` |
| 클러치마스터실린더 | `클러치마스터실린더` + 차종 | `클러치마스터실린더 NF쏘나타` |
| 휠실린더 | `휠실린더` + 차종·위치 | `휠실린더 전 우 마이티` |

이 상태에서 상품명만 무작위로 섞어 나누면 동일한 형식이 학습과 평가 양쪽에 들어간다.

모델이 부품을 구분하는 것이 아니라 해당 공급사의 표기 형식을 외워도 맞힐 수 있어,
성능이 실제보다 높게 측정된다.
따라서 공급사 단위 분할을 함께 사용한다.


단, 공급사 단위로 나누면 공급사가 하나뿐인 카테고리는 학습과 평가 중 한쪽에만 들어간다.

평가 쪽에 들어가면 모델은 그 카테고리를 배운 적이 없어 반드시 틀리게 되므로,
이런 카테고리는 평가에서 제외하고 몇 개를 제외했는지 함께 기록한다.

### 4-2. 무작위 분할

`stratify`를 지정해 학습과 평가의 카테고리 비율을 동일하게 유지한다.

지정하지 않으면 일부 카테고리가 평가에 포함되지 않을 수 있다.

In [24]:
from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42

train_idx, test_idx = train_test_split(
    np.arange(len(data)), test_size=0.2,
    stratify=data["category"], random_state=SEED)

print("학습:", len(train_idx))
print("평가:", len(test_idx))

학습: 8660
평가: 2166


### 4-3. 공급사 단위 분할

`groups`에 공급사를 지정하면 같은 공급사의 상품은 통째로 한쪽에 배정된다.

공급사 단위로 끊기 때문에 비율이 정확히 8:2가 되지는 않는다.

In [25]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx_supplier, test_idx_supplier = next(gss.split(data, groups=data["supplier_id"]))

print("학습:", len(train_idx_supplier))
print("평가:", len(test_idx_supplier))

train_categories = set(data.iloc[train_idx_supplier]["category"])
test_categories = set(data.iloc[test_idx_supplier]["category"])
print("학습에 없는 카테고리:", len(test_categories - train_categories))

학습: 9229
평가: 1597
학습에 없는 카테고리: 0


In [26]:
from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42

train_idx, test_idx = train_test_split(
    np.arange(len(data)), test_size=0.2,
    stratify=data["category"], random_state=SEED)

print("학습:", len(train_idx))
print("평가:", len(test_idx))

학습: 8660
평가: 2166


**분할 결과**

| 방식 | 학습 | 평가 |
|---|---|---|
| 무작위 | 8,660 | 2,166 |
| 공급사 단위 | 9,229 | 1,597 |

SEED 42에서는 학습에 없는 카테고리가 발생하지 않았다.

다만 어떤 공급사가 평가에 배정되는지에 따라 달라질 수 있으므로,
반복 평가 시 매번 확인하여 해당 카테고리를 제외한다.

### 4-4. 분할 결과 저장

분할 결과를 파일로 저장한다.

같은 데이터로 나눠도 실행할 때마다 다르게 섞이면, 2차에서 딥러닝 모델의 성능이 올라갔을 때
모델이 좋아진 것인지 평가 데이터가 쉬워진 것인지 구분할 수 없다.

저장해두면 이후 회차에서도 동일한 학습·평가 데이터를 사용할 수 있다.

In [27]:
import os

os.makedirs("splits", exist_ok=True)

def save_split(names, test_positions, kind, seed):
    fold = pd.Series("train", index=range(len(names)))
    fold.iloc[test_positions] = "test"

    path = f"splits/split_{kind}_seed{seed}.csv"
    pd.DataFrame({"name": names.values, "fold": fold.values}).to_csv(path, index=False)
    print(f"저장: {path}  (학습 {(fold == 'train').sum()}, 평가 {(fold == 'test').sum()})")

save_split(data["name"], test_idx, "random", SEED)
save_split(data["name"], test_idx_supplier, "supplier", SEED)

저장: splits/split_random_seed42.csv  (학습 8660, 평가 2166)
저장: splits/split_supplier_seed42.csv  (학습 9229, 평가 1597)


## 5. 텍스트를 숫자로 변환 (TF-IDF)

모델은 문자를 직접 계산할 수 없으므로 상품명을 숫자 벡터로 바꿔야 한다.
상품명을 글자 조각으로 나눈 뒤, 조각별로 값을 부여한다.

이때 모든 조각을 똑같이 세면 공백이나 `앞`처럼 어디에나 나오는 조각이 판단을 흐린다.
TF-IDF는 드물게 등장하는 조각일수록 높은 값을 주어, 카테고리를 구분하는 데
실제로 도움이 되는 조각을 강조한다.

이 선택이 타당한지는 7-4에서 다른 표현 방식과 비교해 확인한다.

### 5-1. TF-IDF 변환

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

train_names = data.iloc[train_idx]["name"]
test_names = data.iloc[test_idx]["name"]
train_labels = data.iloc[train_idx]["category"]
test_labels = data.iloc[test_idx]["category"]

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), sublinear_tf=True)
# analyzer="char_wb" -> 단어가 아니라 글자 단위로 쪼갬(wb: word boundary)
# ngram_range=(1, 4) — 1글자부터 4글자까지 조각을 모두 사용
# sublinear_tf=True — 등장 횟수에 로그를 씌워 극단값을 완화

X_train = vectorizer.fit_transform(train_names)
X_test = vectorizer.transform(test_names)

print(X_train)

print("학습 행렬:", X_train.shape)
print("평가 행렬:", X_test.shape)

  (np.int32(0), np.int32(0))	0.07912224812441193
  (np.int32(0), np.int32(22162))	0.06438809454318965
  (np.int32(0), np.int32(16127))	0.10973377112089622
  (np.int32(0), np.int32(1563))	0.0835817055260354
  (np.int32(0), np.int32(22322))	0.20795956120368933
  (np.int32(0), np.int32(16128))	0.17249600212778934
  (np.int32(0), np.int32(1593))	0.20795956120368933
  (np.int32(0), np.int32(22323))	0.20795956120368933
  (np.int32(0), np.int32(1594))	0.20795956120368933
  (np.int32(0), np.int32(13814))	0.054465636388102735
  (np.int32(0), np.int32(9414))	0.06514775461219528
  (np.int32(0), np.int32(10782))	0.10242368157287662
  (np.int32(0), np.int32(691))	0.0831196975471269
  (np.int32(0), np.int32(14017))	0.08625817591065954
  (np.int32(0), np.int32(9425))	0.08767802102784616
  (np.int32(0), np.int32(10783))	0.07485403732016134
  (np.int32(0), np.int32(699))	0.09745139549520655
  (np.int32(0), np.int32(14019))	0.10350629853590851
  (np.int32(0), np.int32(9426))	0.10117422942692168
  (np.in

### 5-2. 생성된 글자 조각 확인

In [29]:
print("글자 조각 개수:", len(vectorizer.get_feature_names_out()))
print("예시 조각:", vectorizer.get_feature_names_out()[1000:1020])

글자 조각 개수: 46978
예시 조각: [' 5.7' ' 50' ' 50 ' ' 500' ' 508' ' 50d' ' 50l' ' 51' ' 518' ' 52' ' 52 '
 ' 520' ' 523' ' 524' ' 525' ' 528' ' 529' ' 53' ' 530' ' 532']


### 5-3. IDF 가중치 확인

In [30]:
idf = pd.Series(vectorizer.idf_, index=vectorizer.get_feature_names_out())

check = [" ", "(", "패드", "앞", "쇼바", "캘리", "등속", "써머"]
idf[[c for c in check if c in idf.index]].sort_values()

      1.000000
(     2.076009
패드    3.296943
앞     3.409859
쇼바    4.475598
캘리    5.612238
써머    6.353013
등속    6.570078
dtype: float64

### 5-4. 변환 결과 예시

In [31]:
sample = ["앞등속(좌)"]
vec = vectorizer.transform(sample)

nonzero = vec.nonzero()[1]
feature_names = vectorizer.get_feature_names_out()

result = pd.DataFrame({
    "조각": [feature_names[i] for i in nonzero],
    "값": [vec[0, i] for i in nonzero],
}).sort_values("값", ascending=False)

result.head(50)

,조각,값
17,속(좌),0.278235
16,속(좌,0.269310
13,등속(좌,0.269310
21,앞등속(,0.251948
12,등속(,0.247805
2,앞등,0.247805
3,앞등속,0.247805
20,앞등속,0.247805
19,앞등,0.247805
15,속(,0.247805


## 6. 모델 학습

5단계에서 만든 숫자 벡터를 입력으로 카테고리를 예측하는 모델을 학습한다.

비교 기준선 하나와 머신러닝 모델 세 가지를 사용하며, 모두 동일한 입력을 사용한다.

| 구분 | 방법 | 역할 |
|---|---|---|
| 기준선 | 키워드 규칙 | 머신러닝이 필요한 문제인지 확인 |
| 모델 | 나이브 베이즈 | 가장 단순한 확률 기반 분류 |
| 모델 | 로지스틱 회귀 | 기초 모델 |
| 모델 | 선형 SVM | 고차원 희소 데이터에 적합 |

### 6-1. 규칙 기준선

상품명에 카테고리 이름이 그대로 포함되어 있으면 그 카테고리로 예측하는 단순 규칙이다.

여러 개가 포함된 경우 가장 긴 이름을 선택하고, 없으면 예측하지 않는다.

모델이 아니라 비교 대상이다. 이 규칙만으로 높은 성능이 나온다면
머신러닝을 적용할 이유가 없기 때문에 먼저 측정한다.

In [32]:
categories = sorted(data["category"].unique())

def rule_predict(names):
    predictions = []
    for name in names:
        compact = name.replace(" ", "")
        hits = [c for c in categories if c.replace(" ", "") in compact]
        predictions.append(max(hits, key=len) if hits else "__NONE__")
    return predictions

rule_pred = rule_predict(test_names)

no_prediction = sum(1 for p in rule_pred if p == "__NONE__")
print(f"예측하지 못한 비율: {no_prediction / len(rule_pred):.1%}")

예측하지 못한 비율: 76.7%


### 6-2. 머신러닝 모델 학습

세 모델 모두 동일한 TF-IDF 입력을 사용한다. 차이는 그 숫자를 보고
카테고리를 결정하는 방식에 있다.

- **나이브 베이즈**: 조각별 등장 확률을 곱해 가장 가능성 높은 카테고리를 선택
- **로지스틱 회귀**: 조각마다 가중치를 학습해 점수를 계산
- **선형 SVM**: 카테고리 사이의 경계를 최대한 여유 있게 설정

로지스틱 회귀는 기본 설정(lbfgs)에서 학습에 40초가량 소요되어 `saga`를 사용한다.
성능은 동일하고 속도만 개선된다.

In [33]:
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

models = {
    "나이브 베이즈": ComplementNB(alpha=0.3),
    "로지스틱 회귀": LogisticRegression(C=10, solver="saga", max_iter=300, tol=1e-3),
    "선형 SVM": LinearSVC(C=1.0, max_iter=10000),
}

predictions = {"규칙(카테고리명 포함)": rule_pred}

for name, model in models.items():
    model.fit(X_train, train_labels)
    predictions[name] = model.predict(X_test)
    print(f"{name} 학습 완료")

나이브 베이즈 학습 완료
로지스틱 회귀 학습 완료
선형 SVM 학습 완료


### 6-3. 예측 결과 확인

성능 수치를 계산하기 전에 실제 예측이 어떻게 나왔는지 확인한다.

In [34]:
sample = pd.DataFrame({
    "상품명": test_names.values,
    "정답": test_labels.values,
    "규칙": predictions["규칙(카테고리명 포함)"],
    "로지스틱 회귀": predictions["로지스틱 회귀"],
})

sample.sample(15, random_state=1)

,상품명,정답,규칙,로지스틱 회귀
1547,4PK815 벨트,V벨트,__NONE__,V벨트
1131,6PK2220 벨트,V벨트,__NONE__,V벨트
861,투싼 신형 A/T (두원),라디에이터,__NONE__,라디에이터
573,GOLF 1.8 TDI 05~ (2.3KW),스타터모터,__NONE__,스타터모터
1861,에어필터 S600 5.5 05~13,에어필터,에어필터,에어필터
367,BMW 호환 Radiator 17118592166,라디에이터,__NONE__,라디에이터
1715,뒤패드 CT6 17~,브레이크패드,__NONE__,브레이크패드
201,현대 540 ABS 후드럼,브레이크드럼,__NONE__,브레이크드럼
1651,G CHEROKEE 01~05 (2.0KW),스타터모터,__NONE__,스타터모터
1238,INTAKE MANIFOLD,흡기매니폴드,__NONE__,흡기매니폴드


# 7. 평가

세 가지 지표로 성능을 측정한다.

| 지표 | 의미 | 사용 이유 |
|---|---|---|
| 정확도 | 전체 중 맞힌 비율 | 기본 성능 |
| Macro-F1 | 카테고리별 F1의 평균 | 상품이 적은 카테고리도 동등하게 반영 |
| Top-3 정확도 | 상위 3개 안에 정답이 있는 비율 | 서비스에서 후보를 여러 개 제시하기 때문 |

카테고리마다 가진 상품 개수가 천차만별이므로, 정확도만 보면 상품이 많은 카테고리를 잘 맞히는 것만으로 높은 값이 나온다.

모든 카테고리를 동등하게 반영하는 Macro-F1을 함께 사용한다.

### 7-1. 무작위 분할 성능

In [35]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

labels = sorted(set(test_labels))

def evaluate(pred, model=None, X=None, y=None):
    result = {
        "정확도": accuracy_score(y, pred),
        "Macro-F1": f1_score(y, pred, average="macro", labels=labels, zero_division=0),
        "Top-3 정확도": np.nan,
    }
    if model is not None and hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        top3 = np.argsort(-proba, axis=1)[:, :3]
        result["Top-3 정확도"] = np.mean(
            [true in model.classes_[idx] for true, idx in zip(y, top3)])
    return result

rows = {"규칙(카테고리명 포함)": evaluate(rule_pred, y=test_labels)}
for name, model in models.items():
    rows[name] = evaluate(predictions[name], model, X_test, test_labels)

results = pd.DataFrame(rows).T
results.round(3)

,정확도,Macro-F1,Top-3 정확도
규칙(카테고리명 포함),0.225,0.291,NaN
나이브 베이즈,0.900,0.814,0.950
로지스틱 회귀,0.929,0.883,0.963
선형 SVM,0.934,0.878,NaN


### 7-2. 공급사 단위 분할 성능

처음 보는 공급사의 상품명에도 통하는지 확인한다.
학습에 포함되지 않은 카테고리는 평가에서 제외한다.

In [36]:
train_s = data.iloc[train_idx_supplier]
test_s = data.iloc[test_idx_supplier]

# 학습에 없는 카테고리 제외
known = set(train_s["category"])
excluded = (~test_s["category"].isin(known)).sum()
test_s = test_s[test_s["category"].isin(known)]
print(f"평가 제외: {excluded}건, 평가 대상: {len(test_s)}건")

# 같은 방식으로 변환 (학습 데이터로만 fit)
vec_s = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), sublinear_tf=True)
X_train_s = vec_s.fit_transform(train_s["name"])
X_test_s = vec_s.transform(test_s["name"])

labels_s = sorted(set(test_s["category"]))

rows_s = {}
rule_pred_s = rule_predict(test_s["name"])
rows_s["규칙(카테고리명 포함)"] = {
    "정확도": accuracy_score(test_s["category"], rule_pred_s),
    "Macro-F1": f1_score(test_s["category"], rule_pred_s,
                         average="macro", labels=labels_s, zero_division=0),
    "Top-3 정확도": np.nan,
}

for name, model_class in [("나이브 베이즈", ComplementNB(alpha=0.3)),
                          ("로지스틱 회귀", LogisticRegression(C=10, solver="saga",
                                                          max_iter=300, tol=1e-3)),
                          ("선형 SVM", LinearSVC(C=1.0, max_iter=10000))]:
    model_class.fit(X_train_s, train_s["category"])
    pred_s = model_class.predict(X_test_s)

    row = {
        "정확도": accuracy_score(test_s["category"], pred_s),
        "Macro-F1": f1_score(test_s["category"], pred_s,
                             average="macro", labels=labels_s, zero_division=0),
        "Top-3 정확도": np.nan,
    }
    if hasattr(model_class, "predict_proba"):
        proba = model_class.predict_proba(X_test_s)
        top3 = np.argsort(-proba, axis=1)[:, :3]
        row["Top-3 정확도"] = np.mean(
            [t in model_class.classes_[i] for t, i in zip(test_s["category"], top3)])
    rows_s[name] = row

results_supplier = pd.DataFrame(rows_s).T
results_supplier.round(3)

평가 제외: 0건, 평가 대상: 1597건


,정확도,Macro-F1,Top-3 정확도
규칙(카테고리명 포함),0.147,0.278,NaN
나이브 베이즈,0.843,0.647,0.923
로지스틱 회귀,0.811,0.721,0.894
선형 SVM,0.857,0.746,NaN


### 7-3. 두 분할 비교

In [37]:
compare = pd.DataFrame({
    "A. 무작위": results["정확도"],
    "B. 공급사": results_supplier["정확도"],
})
compare["차이"] = compare["B. 공급사"] - compare["A. 무작위"]
compare.round(3)

,A. 무작위,B. 공급사,차이
규칙(카테고리명 포함),0.225,0.147,-0.078
나이브 베이즈,0.900,0.843,-0.056
로지스틱 회귀,0.929,0.811,-0.118
선형 SVM,0.934,0.857,-0.076


### 7-4. 텍스트 표현 방식 비교

5단계에서 글자 단위 TF-IDF를 선택했다. 그 선택이 타당했는지 표현 방식만 바꿔가며 확인한다.
모델이 달라지면 원인을 구분할 수 없으므로 로지스틱 회귀로 고정하고, 무작위 분할 5회 평균을 사용한다.

비교하는 축은 두 개다.

- **자르는 단위**: 글자 조각(char_wb) / 단어(word)
- **값을 매기는 방식**: TF-IDF / 등장 횟수 / 있음-없음(이진)

In [38]:
# 글자 단위를 선택한 근거가 되는 데이터 특성
names = data["name"]

print(f"공백이 전혀 없는 상품명: {(~names.str.contains(' ')).mean():.1%}")
print(f"상품명 평균 단어 수: {names.str.split().str.len().mean():.1f}개")

tr_idx, te_idx = train_test_split(
    np.arange(len(data)), test_size=0.2,
    stratify=data["category"], random_state=42)

train_words = set(w for s in data.iloc[tr_idx]["name"] for w in s.split())
test_words = [w for s in data.iloc[te_idx]["name"] for w in s.split()]
unseen = np.mean([w not in train_words for w in test_words])

print(f"평가 데이터 단어 중 학습에서 본 적 없는 단어 비율: {unseen:.1%}")

공백이 전혀 없는 상품명: 7.1%
상품명 평균 단어 수: 3.0개
평가 데이터 단어 중 학습에서 본 적 없는 단어 비율: 21.2%


In [39]:
from sklearn.feature_extraction.text import CountVectorizer

methods = {
    "글자 TF-IDF (현재)":          TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), sublinear_tf=True),
    "글자 TF-IDF (sublinear 없음)": TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4)),
    "글자 이진":                   CountVectorizer(analyzer="char_wb", ngram_range=(1, 4), binary=True),
    "글자 횟수":                   CountVectorizer(analyzer="char_wb", ngram_range=(1, 4)),
    "단어 TF-IDF":                TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True),
    "단어 횟수":                   CountVectorizer(analyzer="word", ngram_range=(1, 2)),
}

seeds = [42, 43, 44, 45, 46]
rows = []

for label, vec in methods.items():
    accs, f1s = [], []
    for seed in seeds:
        tr_idx, te_idx = train_test_split(
            np.arange(len(data)), test_size=0.2,
            stratify=data["category"], random_state=seed)
        tr, te = data.iloc[tr_idx], data.iloc[te_idx]

        X_tr = vec.fit_transform(tr["name"])   # 학습으로만 조각 사전을 만들고
        X_te = vec.transform(te["name"])       # 평가는 그 사전으로 변환

        model = LogisticRegression(C=10, solver="saga", max_iter=300, tol=1e-3)
        model.fit(X_tr, tr["category"])
        pred = model.predict(X_te)

        labels = sorted(set(te["category"]))
        accs.append(accuracy_score(te["category"], pred))
        f1s.append(f1_score(te["category"], pred, average="macro",
                            labels=labels, zero_division=0))

    rows.append({
        "표현 방식": label,
        "조각 수": X_tr.shape[1],
        "정확도": np.mean(accs),
        "표준편차": np.std(accs, ddof=1),
        "Macro-F1": np.mean(f1s),
    })
    print(f"완료: {label}")

compare = pd.DataFrame(rows)
compare["정확도"] = compare["정확도"].map(lambda x: f"{x:.1%}")
compare["표준편차"] = compare["표준편차"].map(lambda x: f"{x:.1%}")
compare["Macro-F1"] = compare["Macro-F1"].round(3)

print()
print(compare.to_string(index=False))

완료: 글자 TF-IDF (현재)
완료: 글자 TF-IDF (sublinear 없음)
완료: 글자 이진
완료: 글자 횟수
완료: 단어 TF-IDF
완료: 단어 횟수

                   표현 방식  조각 수   정확도 표준편차  Macro-F1
          글자 TF-IDF (현재) 46895 92.8% 0.2%     0.871
글자 TF-IDF (sublinear 없음) 46895 92.8% 0.2%     0.871
                   글자 이진 46895 92.7% 0.1%     0.868
                   글자 횟수 46895 92.3% 0.3%     0.858
               단어 TF-IDF 18186 86.8% 0.3%     0.804
                   단어 횟수 18186 86.1% 0.6%     0.793


두 결정의 효과가 크게 달랐다.

**성능을 가른 것은 글자 단위 분할이다.** 글자 92.9%, 단어 86.8%로 6.1%p 차이가 난다.
평가 데이터에 나온 단어의 21.2%가 학습에서 한 번도 본 적 없는 단어이고,
상품명 평균 길이가 3단어에 불과하기 때문이다.
단어 단위로 자르면 `6PK2390` 같은 규격 코드는 처음 보는 단어라 그대로 버려지지만,
글자 조각으로 자르면 `PK`가 V벨트 규격 표기라는 것을 다른 상품명에서 배워 적용할 수 있다.
조각 수도 글자 단위가 약 4.7만 개로 단어 단위(약 1.8만 개)의 2.6배여서 판단 근거 자체가 많다.

**TF-IDF 가중의 효과는 작았다.** 글자 단위 안에서 TF-IDF 92.9%, 등장 횟수 92.3%로 0.6%p 차이다.
흔한 조각(공백 1.000)과 드문 조각(등속 6.570)의 무게를 구분해주는 만큼은 도움이 되지만,
상품명이 짧아 같은 조각이 한 상품명 안에 여러 번 나오는 경우가 거의 없다.
있음/없음만 기록하는 이진 방식도 92.7%로 거의 같고,
`sublinear_tf`는 켜도 끄도 정확도 차이가 0.1%p 이내로, 시드 간 편차(±0.2%p)보다 작았다.
즉 TF-IDF에서 실제로 기여하는 부분은 빈도(TF)가 아니라 희소성(IDF) 쪽이다.

글자 단위 TF-IDF를 그대로 유지하되,
성능의 근거는 TF-IDF가 아니라 글자 단위 분할에 있다고 보는 것이 맞다.

> 조각 수는 시드마다 학습 데이터가 달라져 46,895~46,978개 사이에서 변한다.
> 위 표의 값은 마지막 시드 기준이다.

### 7-5. 카테고리 기준값 검증

3-4에서 카테고리당 상품명 **20개**를 학습 대상 기준으로 설정했다. 그 값이 적절한지 확인한다.

기준을 낮추면 더 많은 카테고리를 다룰 수 있지만 학습 예시가 부족한 카테고리가 늘어난다.
반대로 높이면 성능은 오르지만 분류할 수 있는 상품 범위가 줄어든다.
기준값을 바꿔가며 두 값이 어떻게 맞바뀌는지 본다.

시간이 오래 걸리므로 시드 3개(42, 43, 44)로 확인한다.

In [40]:
rows = []

for threshold in [5, 10, 20, 30, 50]:
    keep = counts[counts >= threshold].index
    subset = data_all[data_all["category"].isin(keep)].reset_index(drop=True)

    accs, f1s = [], []
    for seed in [42, 43, 44]:
        a, b = train_test_split(
            np.arange(len(subset)), test_size=0.2,
            stratify=subset["category"], random_state=seed)
        tr, te = subset.iloc[a], subset.iloc[b]

        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), sublinear_tf=True)
        A, B = vec.fit_transform(tr["name"]), vec.transform(te["name"])

        model = LogisticRegression(C=10, solver="saga", max_iter=300, tol=1e-3)
        model.fit(A, tr["category"])
        pred = model.predict(B)

        labels = sorted(set(te["category"]))
        accs.append(accuracy_score(te["category"], pred))
        f1s.append(f1_score(te["category"], pred, average="macro",
                            labels=labels, zero_division=0))

    rows.append({
        "기준값": threshold,
        "카테고리 수": len(keep),
        "상품명 수": len(subset),
        "커버리지": len(subset) / len(data_all),
        "정확도": np.mean(accs),
        "Macro-F1": np.mean(f1s),
    })
    print(f"완료: 기준값 {threshold}")

sweep = pd.DataFrame(rows)
sweep["커버리지"] = sweep["커버리지"].map(lambda x: f"{x:.1%}")
sweep["정확도"] = sweep["정확도"].map(lambda x: f"{x:.1%}")
sweep["Macro-F1"] = sweep["Macro-F1"].round(3)

print()
print(sweep.to_string(index=False))

완료: 기준값 5
완료: 기준값 10
완료: 기준값 20
완료: 기준값 30
완료: 기준값 50

 기준값  카테고리 수  상품명 수  커버리지   정확도  Macro-F1
   5     136  11451 98.8% 90.3%     0.727
  10     101  11207 96.7% 91.9%     0.821
  20      74  10826 93.4% 92.9%     0.878
  30      57  10415 89.8% 93.5%     0.903
  50      42   9812 84.6% 94.8%     0.932


| 기준값 | 카테고리 수 | 상품명 수 | 커버리지 | 정확도 | Macro-F1 | 최소 카테고리의 평가 건수 |
|---|---|---|---|---|---|---|
| 5 | 136 | 11,451 | 98.8% | 90.3% | 0.727 | 1 |
| 10 | 101 | 11,207 | 96.7% | 91.9% | 0.821 | 2 |
| **20** | **74** | **10,826** | **93.4%** | **92.9%** | **0.878** | **4** |
| 30 | 57 | 10,415 | 89.8% | 93.6% | 0.903 | 6 |
| 50 | 42 | 9,812 | 84.6% | 94.8% | 0.932 | 10 |

**주의: 행끼리 성능을 직접 비교해 최고점을 고르면 안 된다.** 기준을 올리면
맞춰야 할 카테고리가 136개에서 42개로 줄어든다. 예시가 적고 어려운 카테고리를
문제에서 제외하는 것이므로 Macro-F1이 오르는 것은 당연하며, 모델이 좋아진 것이 아니다.

이 실험에서 읽어야 할 것은 두 가지다.

**기준값 10 이하에서는 평가가 성립하지 않는다.** 기준값이 t일 때 가장 작은 카테고리의
평가용 상품명은 t의 20%다. 기준값 5에서는 1개이므로 그 카테고리의 F1은 0 또는 1만
나온다. 이런 카테고리가 수십 개 포함된 Macro-F1(0.727)은 성능을 측정한 값이라기보다
우연에 좌우되는 값이다.

**20을 넘어서면 얻는 것이 급격히 줄어든다.** 커버리지 1%p를 포기해 얻는 Macro-F1은
10→20 구간에서 0.017인 반면 20→30 구간에서는 0.007로 절반 이하가 된다.
그 이후로는 커버리지만 계속 줄어든다.

따라서 기준값 20은 이 표에서 최고 성능을 낸 값이라서가 아니라,
3-4에서 세운 요구조건(평가용 상품명 최소 4개 확보)을 만족하면서
평가가 무너지는 구간 밖에 있고 커버리지 93.4%를 유지하기 때문에 선택한 값이다.
20에서 30 사이라면 다른 값도 선택할 수 있다.

> 이 표는 시드 3개 평균이므로 8단계의 5회 반복 결과와 소수점 셋째 자리에서 차이가 있다.

### 7-6. 결과 해석

**머신러닝의 필요성이 확인되었다.** 규칙 기준선은 22.5%에 그쳤고 상품명의 76.7%는
예측조차 하지 못한 반면, 머신러닝 모델은 93%를 기록했다. 카테고리명이 상품명에
포함되지 않는 경우가 대부분이므로 단순 규칙으로는 해결할 수 없는 문제임이 드러났다.

**Top-3 정확도 96.3%는 서비스 적용 가능성을 시사한다.** 후보를 3개 제시하는 방식이라면
대부분의 상품이 정답 카테고리를 포함하게 된다.

**공급사가 바뀌면 성능이 하락한다.** 무작위 분할 92.9%에서 공급사 분할 81.1%로
11.8%p 감소했다(로지스틱 회귀 기준). 모델이 부품 자체보다 공급사별 표기 형식에
일부 의존하고 있음을 의미한다.

| 모델 | 무작위 | 공급사 | 차이 |
|---|---|---|---|
| 나이브 베이즈 | 0.900 | 0.843 | -0.056 |
| 로지스틱 회귀 | 0.929 | 0.811 | **-0.118** |
| 선형 SVM | 0.934 | 0.857 | -0.076 |

특히 로지스틱 회귀의 낙폭이 가장 크다. 가중치를 세밀하게 학습하는 만큼 학습 데이터의
표기 형식에 더 민감하게 맞춰진 것으로 보인다. 반면 Macro-F1은 0.883에서 0.721로
정확도보다 큰 폭으로 하락했는데, 공급사가 1~2곳에 편중된 소수 카테고리가
더 큰 영향을 받은 것으로 해석된다.

# 8. 반복 평가

지금까지의 결과는 한 번의 분할에서 나온 수치다. 분할이 어떻게 되느냐에 따라
성능이 달라질 수 있으므로, 시드를 바꿔 5회 반복하고 평균과 표준편차를 함께 확인한다.

표준편차를 알면 이후 회차에서 성능이 올랐을 때 그것이 의미 있는 개선인지 판단할 수 있다.
편차가 ±0.2%p인데 0.1%p 올랐다면 우연일 가능성이 크기 때문이다.

In [41]:
def run_experiment(seed, kind):
    """지정한 시드와 분할 방식으로 학습·평가를 수행하고 결과를 반환한다."""
    if kind == "random":
        tr_idx, te_idx = train_test_split(
            np.arange(len(data)), test_size=0.2,
            stratify=data["category"], random_state=seed)
    else:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        tr_idx, te_idx = next(gss.split(data, groups=data["supplier_id"]))

    tr, te = data.iloc[tr_idx], data.iloc[te_idx]

    # 학습에 없는 카테고리는 평가에서 제외
    known = set(tr["category"])
    excluded = (~te["category"].isin(known)).sum()
    te = te[te["category"].isin(known)]

    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), sublinear_tf=True)
    X_tr = vec.fit_transform(tr["name"])
    X_te = vec.transform(te["name"])
    y_tr, y_te = tr["category"], te["category"]
    lbl = sorted(set(y_te))

    rows = []

    # 규칙 기준선
    pred = rule_predict(te["name"])
    rows.append({"분할": kind, "시드": seed, "모델": "규칙(카테고리명 포함)",
                 "정확도": accuracy_score(y_te, pred),
                 "Macro-F1": f1_score(y_te, pred, average="macro",
                                      labels=lbl, zero_division=0),
                 "Top-3 정확도": np.nan, "제외": excluded})

    # 머신러닝 모델
    for name, model in [
        ("나이브 베이즈", ComplementNB(alpha=0.3)),
        ("로지스틱 회귀", LogisticRegression(C=10, solver="saga", max_iter=300, tol=1e-3)),
        ("선형 SVM", LinearSVC(C=1.0, max_iter=10000)),
    ]:
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        row = {"분할": kind, "시드": seed, "모델": name,
               "정확도": accuracy_score(y_te, pred),
               "Macro-F1": f1_score(y_te, pred, average="macro",
                                    labels=lbl, zero_division=0),
               "Top-3 정확도": np.nan, "제외": excluded}
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_te)
            top3 = np.argsort(-proba, axis=1)[:, :3]
            row["Top-3 정확도"] = np.mean(
                [t in model.classes_[i] for t, i in zip(y_te, top3)])
        rows.append(row)

    return rows

In [42]:
SEEDS = [42, 43, 44, 45, 46]

all_rows = []
for seed in SEEDS:
    for kind in ["random", "supplier"]:
        all_rows += run_experiment(seed, kind)
        print(f"완료: seed {seed}, {kind}")

repeated = pd.DataFrame(all_rows)
repeated.to_csv("results_per_seed.csv", index=False)
print("\n저장: results_per_seed.csv")

완료: seed 42, random
완료: seed 42, supplier
완료: seed 43, random
완료: seed 43, supplier
완료: seed 44, random
완료: seed 44, supplier
완료: seed 45, random
완료: seed 45, supplier
완료: seed 46, random
완료: seed 46, supplier

저장: results_per_seed.csv


### 8-1. 평균과 표준편차

In [43]:
order = ["규칙(카테고리명 포함)", "나이브 베이즈", "로지스틱 회귀", "선형 SVM"]

for kind, title in [("random", "A. 무작위 분할"), ("supplier", "B. 공급사 단위 분할")]:
    print(f"\n[{title}]  {len(SEEDS)}회 반복")
    subset = repeated[repeated["분할"] == kind]
    for m in order:
        g = subset[subset["모델"] == m]
        acc = f"{g['정확도'].mean():.1%} ± {g['정확도'].std():.1%}"
        f1 = f"{g['Macro-F1'].mean():.3f} ± {g['Macro-F1'].std():.3f}"
        t3 = g["Top-3 정확도"].mean()
        t3s = f"{t3:.1%}" if not np.isnan(t3) else "-"
        print(f"  {m:<20} 정확도 {acc:<18} Macro-F1 {f1:<18} Top-3 {t3s}")


[A. 무작위 분할]  5회 반복
  규칙(카테고리명 포함)         정확도 22.8% ± 0.8%       Macro-F1 0.301 ± 0.024      Top-3 -
  나이브 베이즈              정확도 90.0% ± 0.4%       Macro-F1 0.812 ± 0.006      Top-3 95.1%
  로지스틱 회귀              정확도 92.9% ± 0.2%       Macro-F1 0.872 ± 0.011      Top-3 96.3%
  선형 SVM               정확도 93.0% ± 0.2%       Macro-F1 0.866 ± 0.008      Top-3 -

[B. 공급사 단위 분할]  5회 반복
  규칙(카테고리명 포함)         정확도 16.7% ± 5.7%       Macro-F1 0.265 ± 0.031      Top-3 -
  나이브 베이즈              정확도 79.0% ± 10.7%      Macro-F1 0.649 ± 0.104      Top-3 85.0%
  로지스틱 회귀              정확도 76.9% ± 9.7%       Macro-F1 0.690 ± 0.077      Top-3 82.6%
  선형 SVM               정확도 81.2% ± 9.3%       Macro-F1 0.714 ± 0.097      Top-3 -


In [44]:
# 선형 SVM Top-3 정확도 (확률 대신 카테고리별 점수 순위 사용)
svm_top3 = []
for seed in SEEDS:
    tr_idx, te_idx = train_test_split(
        np.arange(len(data)), test_size=0.2,
        stratify=data["category"], random_state=seed)
    tr, te = data.iloc[tr_idx], data.iloc[te_idx]

    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), sublinear_tf=True)
    X_tr, X_te = vec.fit_transform(tr["name"]), vec.transform(te["name"])

    svm = LinearSVC(C=1.0, max_iter=10000).fit(X_tr, tr["category"])
    scores = svm.decision_function(X_te)          # 카테고리별 점수
    top3 = np.argsort(-scores, axis=1)[:, :3]     # 점수 상위 3개
    svm_top3.append(np.mean([t in svm.classes_[i] for t, i in zip(te["category"], top3)]))

print(f"선형 SVM Top-3 (무작위 분할 5회): {np.mean(svm_top3):.1%} ± {np.std(svm_top3, ddof=1):.1%}")

선형 SVM Top-3 (무작위 분할 5회): 96.2% ± 0.2%


### 8-2. 시드별 변동 확인

공급사 분할의 편차가 큰지 시드별 수치를 직접 확인한다.

In [45]:
pivot = repeated[repeated["모델"] == "로지스틱 회귀"].pivot(
    index="시드", columns="분할", values="정확도")
pivot["제외 건수"] = repeated[
    (repeated["모델"] == "로지스틱 회귀") & (repeated["분할"] == "supplier")
]["제외"].values
pivot.round(3)

분할,random,supplier,제외 건수
시드,,,
42,0.929,0.812,0
43,0.928,0.845,0
44,0.932,0.612,72
45,0.926,0.835,0
46,0.929,0.739,0


### 8-3. 반복 평가 결과

| 분할 | 모델 | 정확도 | Macro-F1 | Top-3 |
|---|---|---|---|---|
| A. 무작위 | 규칙 | 22.8% ± 0.8% | 0.301 | - |
| A. 무작위 | 나이브 베이즈 | 90.0% ± 0.4% | 0.812 | 95.1% |
| A. 무작위 | 로지스틱 회귀 | 92.9% ± 0.2% | 0.872 | 96.3% |
| A. 무작위 | 선형 SVM | 93.0% ± 0.2% | 0.866 | - |
| B. 공급사 | 규칙 | 16.7% ± 5.7% | 0.265 | - |
| B. 공급사 | 나이브 베이즈 | 79.0% ± 10.7% | 0.649 | 85.0% |
| B. 공급사 | 로지스틱 회귀 | 76.8% ± 9.7% | 0.689 | 82.6% |
| B. 공급사 | 선형 SVM | 81.2% ± 9.3% | 0.714 | - |

**무작위 분할은 안정적이다.** 표준편차가 0.2%p로, 이후 회차에서 1%p 이상 개선되면
의미 있는 변화로 판단할 수 있다.

**공급사 분할은 편차가 크다.** 표준편차가 10%p에 달해 단일 수치를 신뢰하기 어렵다.
시드별 결과를 보면 원인이 드러난다.

| 시드 | 무작위 | 공급사 | 학습에 없어 제외된 건수 |
|---|---|---|---|
| 42 | 92.9% | 81.1% | 0 |
| 43 | 92.8% | 84.5% | 0 |
| 44 | 93.2% | **61.2%** | **72** |
| 45 | 92.6% | 83.4% | 0 |
| 46 | 92.9% | 73.9% | 0 |

시드 44에서 정확도가 61.2%로 급락했으며, 이때만 학습에 없는 카테고리 72건이 발생했다.
특정 공급사가 평가에 배정되면서 해당 공급사에만 존재하던 카테고리가 학습에서 누락된 것이다.

이는 4단계에서 확인한 공급사 편중(단일 공급사 카테고리 4개, 두 공급사 카테고리 9개)이
평가 결과에 실제로 영향을 미침을 보여준다. 따라서 공급사 분할 결과는 단일 수치가 아니라
반복 평균과 편차로 보고해야 한다.

또한 공급사 분할에서는 선형 SVM(81.2%)이 로지스틱 회귀(76.8%)보다 높다.
무작위 분할에서는 두 모델이 거의 같았으므로(93.0% / 92.9%),
이 조건에서 모델 간 우열을 단정하기는 어렵다.

# 9. 오류 분석과 실제 적용

성능 수치만으로는 모델이 무엇을 못하는지 알 수 없다.

틀린 예측을 유형별로 분석하고, 학습된 모델을 실제 미분류 상품에 적용해 본다.

### 9-1. 자주 혼동하는 카테고리

In [46]:
errors = pd.DataFrame({
    "상품명": test_names.values,
    "정답": test_labels.values,
    "예측": predictions["로지스틱 회귀"],
})
errors = errors[errors["정답"] != errors["예측"]]

print(f"오답: {len(errors)}건 ({len(errors)/len(test_names):.1%})")

errors.groupby(["정답", "예측"]).size().sort_values(ascending=False).head(15)

오답: 154건 (7.1%)


정답         예측     
냉각팬모터      라디에이터      7
컨덴서휀       컨덴서        5
라디에이터      컨덴서        4
써모스탯       라디에이터호스    4
타이밍벨트      냉각팬모터      3
볼트/너트      라디에이터호스    3
쇼크업소버      브레이크드럼     2
브레이크드럼     쇼크업소버      2
볼트/너트      로어암        2
엔진오일필터하우징  오일필터       2
범퍼         라디에이터호스    2
배터리        라디에이터호스    2
알터네이터      냉각팬모터      2
라디에이터호스    워터펌프       2
컨덴서        라디에이터      2
dtype: int64

### 9-2. 오류 사례 확인

실제 오답 사례를 확인해 원인을 분류한다.

In [47]:
errors.sample(25, random_state=1)

,상품명,정답,예측
882,팬-플로어 스플래쉬 쉴드,언더커버,브레이크패드
530,카니발(730)베어링(NSK-B696),아이들베어링,브레이크드럼
1005,"BATTERY CABLE, NEGATIVE",배터리,라디에이터호스
2048,SEAL KIT-EX PTU CO,리데나,라디에이터호스
510,COVER CRANKSHAFT REAR,리데나,범퍼
1595,라세티,냉각팬모터,브레이크드럼
1448,하우징,라디에이터호스,써모스탯
175,메가트럭,알터네이터,컨덴서
780,"CAP SUB-ASSY,",라디에이터호스,쇼크업소버
679,"카니발,테라칸2.9 텃취(750)",아이들베어링,V벨트


### 9-3. 카테고리별 성능

Macro-F1을 끌어내리는 카테고리를 확인한다.

In [48]:
from sklearn.metrics import f1_score

per_class = pd.Series(
    f1_score(test_labels, predictions["로지스틱 회귀"],
             average=None, labels=labels, zero_division=0),
    index=labels
).sort_values()

count_in_test = test_labels.value_counts()

pd.DataFrame({
    "F1": per_class.round(3),
    "평가 건수": count_in_test.reindex(per_class.index),
}).head(15)

,F1,평가 건수
냉각팬모터,0.640,27
라디에이터호스,0.707,41
흡기매니폴드,0.741,14
볼트/너트,0.778,21
컨덴서,0.804,48
타이밍벨트,0.850,22
냉각수탱크,0.857,11
써모스탯,0.857,27
라디에이터,0.880,87
헤드커버,0.882,17


### 9-4. 오류 분석 결과

오답 153건(7.1%)을 분석한 결과 세 가지 유형으로 구분된다.

**① 냉각 계통 카테고리 간 혼동**

| 정답 | 예측 | 건수 |
|---|---|---|
| 냉각팬모터 | 라디에이터 | 7 |
| 컨덴서휀 | 컨덴서 | 5 |
| 라디에이터 | 컨덴서 | 4 |
| 써모스탯 | 라디에이터호스 | 4 |
| 컨덴서 | 라디에이터 | 2 |

라디에이터·컨덴서·냉각팬모터·컨덴서휀·써모스탯은 모두 냉각 계통 부품으로 상품명이 유사하다.
`올뉴카렌스`처럼 차종만 기재된 경우 구분이 불가능하다.
카테고리 경계 자체가 모호한 사례로, 통합 여부를 검토할 필요가 있다.

**② 차종만 기재된 상품명**

`라세티`, `A3`, `메가트럭`, `K5,YF,LF`, `올뉴카렌스` 등 부품 정보가 전혀 없다.
사람도 판별할 수 없으므로 모델 성능의 구조적 상한에 해당한다.

**③ 범용 명칭**

`하우징`, `슬리브`, `SEAL SET`, `CAP SUB-ASSY,`, `REP WIRE FOR B` 등
부품 유형을 특정할 수 없는 이름이다.
3단계에서 확인한 카테고리 충돌(`CLIP`, `BRACKET`)과 같은 성격이다.

**카테고리별 성능**

| 카테고리 | F1 | 평가 건수 |
|---|---|---|
| 컨덴서휀 | 0.400 | 7 |
| 몰딩/트림 | 0.533 | 9 |
| 터보차저/인터쿨러 | 0.545 | 6 |
| 리데나 | 0.571 | 5 |
| 도어 | 0.588 | 10 |
| 냉각팬모터 | 0.640 | 27 |

하위 카테고리는 대부분 평가 건수가 10건 이하다. Macro-F1이 정확도보다 낮게 나타나는 원인이다.
다만 냉각팬모터는 평가 건수가 27건으로 적지 않은데도 F1이 0.640에 그치는데,
이는 건수 부족이 아니라 ①의 카테고리 경계 모호성 때문으로 보인다.

### 9-5. 미분류 상품에 적용

카테고리가 지정되지 않았거나 "기타"로 남은 상품에 학습된 모델을 적용한다.
상품명이 재고 메모인 경우는 제외한다.

예측 확신도가 낮은 상품은 자동 분류하지 않고 검토 대상으로 분리한다.
학습에 포함되지 않은 카테고리에 속할 가능성이 있기 때문이다.

In [49]:
# 전체 학습 데이터로 다시 학습 (평가용 분할이 아닌 전체 사용)
final_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), sublinear_tf=True)
X_all = final_vec.fit_transform(data["name"])

final_model = LogisticRegression(C=10, solver="saga", max_iter=300, tol=1e-3)
final_model.fit(X_all, data["category"])

# 적용 대상 추출
target = df[df["split_group"].isin(["unlabeled", "etc"])].copy()
target = target[~target["name_clean"].str.contains(NOTICE)]
print("적용 대상:", len(target))

proba = final_model.predict_proba(final_vec.transform(target["name_clean"]))
target["예측"] = final_model.classes_[proba.argmax(axis=1)]
target["확신도"] = proba.max(axis=1).round(3)

target[["name_clean", "category", "예측", "확신도"]] \
    .sort_values("확신도", ascending=False).head(20)

적용 대상: 804


,name_clean,category,예측,확신도
3598,링크,,스태빌라이저링크,1.0
3474,코일,,점화코일,1.0
3510,점화코일,,점화코일,1.0
3480,점화코일,,점화코일,1.0
1568,앞 디스크 1장,,디스크로터,1.0
3479,점화코일,,점화코일,1.0
3477,점화코일,,점화코일,1.0
1575,앞 디스크,,디스크로터,1.0
3476,점화코일,,점화코일,1.0
3475,점화코일,,점화코일,1.0


### 9-6. 확신도 기준별 처리 가능 범위

In [50]:
for threshold in [0.5, 0.7, 0.8, 0.9]:
    auto = (target["확신도"] >= threshold).sum()
    print(f"확신도 {threshold} 이상: {auto}건 자동 분류 "
          f"({auto/len(target):.1%}), 검토 대상 {len(target)-auto}건")

확신도 0.5 이상: 698건 자동 분류 (86.8%), 검토 대상 106건
확신도 0.7 이상: 670건 자동 분류 (83.3%), 검토 대상 134건
확신도 0.8 이상: 637건 자동 분류 (79.2%), 검토 대상 167건
확신도 0.9 이상: 492건 자동 분류 (61.2%), 검토 대상 312건


In [51]:
target[["id", "name", "category", "예측", "확신도"]] \
    .sort_values("확신도", ascending=False) \
    .to_csv("predictions_unlabeled.csv", index=False)
print("저장: predictions_unlabeled.csv")

저장: predictions_unlabeled.csv


In [52]:
# 중간 확신도 구간
target[(target["확신도"] >= 0.7) & (target["확신도"] < 0.95)][
    ["name_clean", "예측", "확신도"]].head(20)

,name_clean,예측,확신도
141,INA 아이들러 어셈블리,아이들베어링,0.866
143,PULLEY IDLER,아이들베어링,0.931
144,BELT SERPENTINE,V벨트,0.735
146,풀리 INA 532055610,아이들베어링,0.705
156,외벨트 6DPK1697,V벨트,0.935
160,6PK2390,V벨트,0.937
193,TENSIONER,텐셔너,0.918
195,TENSIONER,텐셔너,0.918
200,THERMOSTAT ASSY,써모스탯,0.927
201,THERMOSTAT ASSY,써모스탯,0.927


### 9-7. 적용 결과

미분류·기타 상품 중 재고 메모를 제외한 804건에 모델을 적용하였다.

| 확신도 기준 | 자동 분류 | 검토 대상 |
|---|---|---|
| 0.5 이상 | 698건 (86.8%) | 106건 |
| 0.7 이상 | 670건 (83.3%) | 134건 |
| 0.8 이상 | 636건 (79.1%) | 168건 |
| 0.9 이상 | 492건 (61.2%) | 312건 |

확신도 0.8을 기준으로 하면 약 79%를 자동 분류하고 나머지는 담당자 검토로 넘길 수 있다.

**중간 확신도 구간의 분류 사례**

확신도가 1.0에 가까운 상품은 상품명이 카테고리명과 거의 일치하는 경우로,
규칙만으로도 분류 가능하다. 모델의 실질적 가치는 중간 구간에서 드러난다.

| 상품명 | 예측 카테고리 | 확신도 |
|---|---|---|
| `스타터` | 스타터모터 | 0.840 |
| `발전기` | 알터네이터 | 0.825 |
| `외벨트 6DPK1697` | V벨트 | 0.937 |
| `6PK2390` | V벨트 | 0.938 |
| `PULLEY IDLER` | 아이들베어링 | 0.931 |
| `THERMOSTAT ASSY` | 써모스탯 | 0.927 |

`스타터`→스타터모터, `발전기`→알터네이터처럼 현장 용어와 표준 카테고리명이 다른 경우,
`PULLEY IDLER`처럼 영어로 기재된 경우, `6PK2390`처럼 부품명 없이 규격 코드만 있는 경우까지
분류하고 있다. 모두 카테고리명 포함 여부를 보는 규칙으로는 처리할 수 없는 사례다.

In [53]:
import numpy as np
import pandas as pd

# 표지 예시 상품명의 추천 상위 3개
examples = ["6PK2390", "앞등속(좌)", "발전기"]

X_ex = final_vec.transform(examples)
proba_ex = final_model.predict_proba(X_ex)
top3_idx = np.argsort(-proba_ex, axis=1)[:, :3]

rows = []
for name, idx, p in zip(examples, top3_idx, proba_ex):
    for rank, j in enumerate(idx, start=1):
        rows.append({"상품명": name, "순위": rank,
                     "카테고리": final_model.classes_[j],
                     "확신도": round(p[j], 3)})

pd.DataFrame(rows)

,상품명,순위,카테고리,확신도
0,6PK2390,1,V벨트,0.937
1,6PK2390,2,알터네이터,0.004
2,6PK2390,3,점화플러그,0.003
3,앞등속(좌),1,드라이브샤프트,0.908
4,앞등속(좌),2,타이로드엔드,0.009
5,앞등속(좌),3,브레이크패드,0.005
6,발전기,1,알터네이터,0.825
7,발전기,2,브레이크드럼,0.012
8,발전기,3,쇼크업소버,0.008
